In [ ]:
import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# -----------------------------
# CONFIGURACIÓN GENERAL
# -----------------------------
BASE_PATH = r"D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad4"
COMBINED_FILES = [os.path.join(BASE_PATH, f"combined_data_{i}.txt") for i in range(1, 5)]
MOVIE_TITLES = os.path.join(BASE_PATH, "movie_titles.csv")
OUT_DIR = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# CONFIGURAR SPARK
# -----------------------------
spark = SparkSession.builder \
    .appName("NetflixRecommenderSparkFull") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

# -----------------------------
# PARSER DISTRIBUIDO DE ARCHIVOS COMBINADOS
# -----------------------------
def parse_combined_lines(lines):
    """Convierte las líneas de combined_data_X.txt en tuplas (user, movie, rating)."""
    movie_id = None
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.endswith(":"):
            try:
                movie_id = int(s[:-1])
            except ValueError:
                continue
        else:
            parts = s.split(",")
            if len(parts) == 3:
                try:
                    user = int(parts[0])
                    rating = float(parts[1])
                    yield (user, movie_id, rating)
                except ValueError:
                    continue

def load_data_spark(files):
    """Carga los combined_data_X.txt usando Spark en modo distribuido."""
    rdd = spark.sparkContext.textFile(",".join(files))
    parsed = rdd.mapPartitions(parse_combined_lines)
    df = spark.createDataFrame(parsed, ["UserID", "MovieID", "Rating"])
    print(f"Total registros: {df.count():,}")
    print(f"Usuarios únicos: {df.select('UserID').distinct().count():,}")
    print(f"Películas únicas: {df.select('MovieID').distinct().count():,}")
    return df

# -----------------------------
# MATRIZ USUARIO-PELÍCULA (para similitud local)
# -----------------------------
def build_matrix(df_pd: pd.DataFrame):
    """Crea matriz dispersa usuario-película."""
    user_map = {u: i for i, u in enumerate(sorted(df_pd["UserID"].unique()))}
    movie_map = {m: i for i, m in enumerate(sorted(df_pd["MovieID"].unique()))}
    rows = df_pd["UserID"].map(user_map)
    cols = df_pd["MovieID"].map(movie_map)
    vals = df_pd["Rating"].astype(float)
    R = csr_matrix((vals, (rows, cols)), shape=(len(user_map), len(movie_map)))
    return R, user_map, movie_map

def top_k_similar(sim_vector, k=5):
    """Devuelve índices y pesos de los k más similares."""
    sim_vector = sim_vector.copy()
    sim_vector[np.isnan(sim_vector)] = 0
    top_idx = np.argsort(sim_vector)[::-1][:k]
    top_vals = sim_vector[top_idx]
    return top_idx, top_vals

def recommend_user_based(R, user_map, movie_map, user_id, k=10, n_recs=10):
    """Filtrado colaborativo basado en usuarios (solo local)."""
    R_dense = R.toarray()
    user_index = user_map[user_id]
    user_sim = cosine_similarity(R_dense, R_dense[user_index].reshape(1, -1)).ravel()
    user_sim[user_index] = 0
    top_users, weights = top_k_similar(user_sim, k=k)
    numerator = np.dot(weights, R_dense[top_users])
    denominator = np.abs(weights).sum() + 1e-9
    user_ratings = numerator / denominator
    seen = set(np.where(R_dense[user_index] > 0)[0])
    candidates = [(i, score) for i, score in enumerate(user_ratings) if i not in seen]
    top = sorted(candidates, key=lambda x: x[1], reverse=True)[:n_recs]
    inv_movie_map = {v: k for k, v in movie_map.items()}
    return [(inv_movie_map[i], float(score)) for i, score in top]

def recommend_item_based(R, user_map, movie_map, user_id, k=10, n_recs=10):
    """Filtrado colaborativo basado en ítems (solo local)."""
    R_dense = R.toarray()
    user_index = user_map[user_id]
    user_ratings = R_dense[user_index]
    item_sim = cosine_similarity(R_dense.T)
    seen_idx = np.where(user_ratings > 0)[0]
    if seen_idx.size == 0:
        return []
    numer = item_sim[:, seen_idx] @ user_ratings[seen_idx]
    denom = np.abs(item_sim[:, seen_idx]).sum(axis=1) + 1e-9
    scores = numer / denom
    seen = set(seen_idx.tolist())
    candidates = [(i, scores[i]) for i in range(len(scores)) if i not in seen]
    top = sorted(candidates, key=lambda x: x[1], reverse=True)[:n_recs]
    inv_movie_map = {v: k for k, v in movie_map.items()}
    return [(inv_movie_map[i], float(np.clip(s, 1.0, 5.0))) for i, s in top]

# -----------------------------
# ENTRENAMIENTO Y RECOMENDACIÓN SPARK
# -----------------------------
def als_training(df):
    """Entrena modelo ALS distribuido en Spark y evalúa RMSE."""
    train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
    als = ALS(
        userCol="UserID",
        itemCol="MovieID",
        ratingCol="Rating",
        rank=15,
        maxIter=10,
        regParam=0.1,
        coldStartStrategy="drop",
        nonnegative=True
    )
    model = als.fit(train_df)
    predictions = model.transform(test_df)
    evaluator = RegressionEvaluator(metricName="rmse", labelCol="Rating", predictionCol="prediction")
    rmse = evaluator.evaluate(predictions)
    print(f"\n RMSE del modelo ALS (Spark): {rmse:.4f}\n")
    return model

# -----------------------------
# FUNCIÓN PRINCIPAL
# -----------------------------
def main():
    print("=== Cargando datos en Spark ===")
    df_spark = load_data_spark(COMBINED_FILES)

    # Entrenamos ALS distribuido
    print("=== Entrenando modelo ALS ===")
    als_model = als_training(df_spark)

    # Elegimos usuario de ejemplo
    example_user = df_spark.select("UserID").groupBy("UserID").count().orderBy(F.desc("count")).first()["UserID"]
    print(f"Ejemplo de usuario: {example_user}")

    # Recomendaciones desde Spark (ALS)
    user_recs_spark = als_model.recommendForAllUsers(10)
    recs_user_df = user_recs_spark.filter(F.col("UserID") == example_user).toPandas()

    # Si quieres probar también tus métodos originales en local:
    print("=== Muestra reducida a 3000 usuarios para cálculo local ===")
    sample_pd = df_spark.limit(200000).toPandas()  # puedes ajustar este número
    R, user_map, movie_map = build_matrix(sample_pd)
    user_recs = recommend_user_based(R, user_map, movie_map, example_user, k=10, n_recs=10)
    item_recs = recommend_item_based(R, user_map, movie_map, example_user, k=10, n_recs=10)

    # Cargar títulos de películas
    if os.path.exists(MOVIE_TITLES):
        titles = pd.read_csv(
            MOVIE_TITLES, header=None,
            names=["MovieID", "Year", "Title"],
            encoding="ISO-8859-1",
            engine="python", on_bad_lines="skip", sep=","
        )
        user_df = pd.DataFrame(user_recs, columns=["MovieID", "PredScore"]).merge(titles, on="MovieID", how="left")
        item_df = pd.DataFrame(item_recs, columns=["MovieID", "PredScore"]).merge(titles, on="MovieID", how="left")
    else:
        user_df = pd.DataFrame(user_recs, columns=["MovieID", "PredScore"])
        item_df = pd.DataFrame(item_recs, columns=["MovieID", "PredScore"])

    # Guardar resultados
    user_out = os.path.join(OUT_DIR, f"user_based_{example_user}.csv")
    item_out = os.path.join(OUT_DIR, f"item_based_{example_user}.csv")
    spark_out = os.path.join(OUT_DIR, f"als_based_{example_user}.csv")

    user_df.to_csv(user_out, index=False)
    item_df.to_csv(item_out, index=False)
    recs_user_df.to_csv(spark_out, index=False)

    print(f"\nTop-10 (User-based) guardado en: {user_out}")
    print(f"Top-10 (Item-based) guardado en: {item_out}")
    print(f"Top-10 (ALS Spark) guardado en: {spark_out}")
    print("\n Proceso completado exitosamente")

# -----------------------------
# EJECUCIÓN
# -----------------------------
if __name__ == "__main__":
    main()
